## Questão 1 - Alterando Brilho e Contraste
Implemente uma função que altere o brilho e/ou o contraste de uma imagem
colorida. A funcao deve ter como entrada:
- uma imagem colorida ou preta e branca, com valores $r,g,b$ no intervalo $[0,1]$.
- o valor da alteração do brilho $\beta \in [0,1]$.
- o valor da alteração do contraste $k \in [0,1]$.

O brilho deve ser alterado de forma aditiva, isto é, o pixel com cor $c = (r,g,b)$ deve passar para $c = (r + \beta,g + \beta,b + \beta)$ (é a mesma cor, mas menor saturacão - foi misturada com um pouco de cor branca).

O contraste deve ser alterado de forma multiplicativa, isto é, o pixel com cor $c = (r,g,b)$ deve passar para $c = (k(r − \overline{r}) + \overline{r}, k(g − \overline{g}) + \overline{g}, k(b − \overline{b}) + \overline{b})$.
Teste com a imagem PoucoContraste.png ou outra imagem de sua preferência.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# O imread carrega a imagem. Por padrão, o formato é BGR (Azul, Verde, Vermelho), e não RGB.
# imagem = cv2.imread("imagem/snoopy.jpg")
# imagem = cv2.imread("imagem/reliquias.png")

# preto e branco
imagem = cv2.imread("imagem/PoucoContraste.png", cv2.IMREAD_GRAYSCALE)

def adequar_imagem(imagem, t):
    """
    t=1: Converte a imagem de [0, 255] (uint8) para [0.0, 1.0] (float32)
    t=0: Converte a imagem de [0.0, 1.0] (float32) para [0, 255] (uint8)
    """
    img = np.copy(imagem)
    
    if t == 1:
        # Ida: 0 a 255 -> 0.0 a 1.0
        return img.astype(np.float32) / 255.0
        
    elif t == 0:
        # Volta: 0.0 a 1.0 -> 0 a 255
        return np.clip(img * 255.0, 0, 255).astype(np.uint8)
    else:
        raise ValueError("Parâmetro t deve ser 0 ou 1")
    
def alterar_brilho_contraste(imagem, beta, k):
    """
    beta: brilho pertencente ao intervalo [-1,1]
    k: contraste pertencente ao intervalo [0,1]
    """
    # img colorida no np é uma matriz 3d, onde eixo0 = altura, eixo1 = largura, eixo2: canais de cor BGR
    img_modificada = np.copy(imagem)
    
    # Usar axis=(0, 1) para calcular a média ao longo dos eixos de altura e largura, mantendo os canais de cor separados
    # Faz com que funcione tanto para imagens PB quanto coloridas
    if img_modificada.ndim == 2:
        media = np.mean(img_modificada)
    else:
        media = np.mean(img_modificada, axis=(0, 1))

    img_contraste = k * (img_modificada - media) + media
    img_brilho = img_contraste + beta
    

    # Garante que os limites não saiam de [0, 1]
    img_final = np.clip(img_brilho, 0.0, 1.0)
    

    return img_final

if imagem is None:
    print("Erro: A imagem não foi encontrada.")
else:
    imagem_uint8 = np.copy(imagem) 
    
    if imagem.dtype == np.uint8:
        imagem = adequar_imagem(imagem, t=1)

    parametros_teste = [
        (0, 1.0),     # original
        (0.3, 1.0),   # apenas brilho positivo
        (-0.3, 1.0),  # apenas brilho negativo
        (0.0, 0.5),   # apenas redução de contraste
        (0.3, 0.5),   # brilho positivo + menos contraste
        (-0.3, 0.5),  # brilho negativo + menos contraste
        (0.0, 0.1),   # contraste quase nulo
        (0.5, 0.8),   # muito brilho + contraste levemente reduzido
        (-0.5, 0.8)   # muito escuro + contraste levemente reduzido
    ]

    # fig é a figura geral que contém todos os subplots
    # axes é uma matriz 2D de subplots, onde axes[linha, coluna] acessa cada subplot individualmente
    # Nesse caso, criamos uma grade de 3x3 subplots para acomodar as 9 combinações de parâmetros que queremos testar
    # 15x15 é o tamanho total da figura, em polegadas, garantindo que cada subplot tenha espaço suficiente para exibir 
    # a imagem e o título
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))

    for i, (beta, k) in enumerate(parametros_teste):
        # Calcula a linha e coluna do subplot com base no índice i, ex: se i=4, linha=1 e coluna=1, ou seja, o subplot do meio; 
        # se i=7, linha=2 e coluna=1, ou seja, o subplot da última linha e segunda coluna

        linha = i // 3
        coluna = i % 3

        # Acesso ao subplot específico para plotar a imagem modificada
        ax = axes[linha, coluna]
        
        img = alterar_brilho_contraste(imagem, beta, k)
        
        # exibe a imagem modificada no subplot correspondente, garantindo que cada combinação de beta e k seja 
        # mostrada em um subplot diferente
        
        # trabalhando com imagens coloridas, é necessário converter de BGR para RGB para exibir corretamente com o Matplotlib,
        # já que o OpenCV usa BGR por padrão, enquanto o Matplotlib espera RGB
        if img.ndim == 3:
            img_plot = img[:, :, ::-1]   # BGR -> RGB
            ax.imshow(img_plot)
        else:
            # para imagens em escala de cinza, o cmap='gray' garante que sejam exibidas corretamente, 
            # e vmin/vmax definem os limites de intensidade para a exibição
            # caso não haja esses parâmetros, o Matplotlib pode tentar adivinhar os limites de intensidade, 
            # o que pode resultar em uma imagem muito escura ou muito clara, dependendo dos valores presentes na imagem
            # no caso, ele tomaria o zero como o valor mínimo e o maior valor presente na imagem como o máximo, o que pode 
            # não ser ideal para imagens com pouco contraste
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)

        ax.set_title(f"Beta: {beta}, k: {k}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## Anotações importantes:
A lógica do contraste gira em torno da distância entre a intensidade de cada pixel e a média de brilho da imagem.
- $k>1$: Aumenta as distâncias da média; Mais contraste, realçando detalhes claros e escuros; As barras se espalham para as bordas (0 e 1);
- $k=1$: Sem alteração;
- $k<1$: Diminui as distâncias da média; Menos contraste, com a imagem ficando "lavada" ou cinzenta; As barras encolhem em direção ao centro;

O termo $(pixel - \text{média})$ identifica o quão longe o pixel está do "cinza médio". O $k$ então decide se essa distância deve ser ampliada ou reduzida. No final, somamos a $\text{média}$ novamente para garantir que a imagem não fique apenas preta ou branca, mas que mantenha sua referência de brilho original. 

## Questão 2 - Histograma
Faça o histograma da imagem original, a imagem com brilho alterado e a imagem com contraste alterado usando a função desenvolvida em (1). 

Compare com o histograma da imagem com brilho e contraste alterado da função:

$\texttt{cv2.convertScaleAbs(img, alpha=contraste, beta=brilho)}$

A sua função implementa o mesmo conceito que a função do OpenCV?

In [ ]:
imagem = cv2.imread("imagem/PoucoContraste.png", cv2.IMREAD_GRAYSCALE)

if imagem.dtype == np.uint8:
        imagem = adequar_imagem(imagem, t=1)

parametros_teste = [
    (0.0, 1.0),   # original
    (0.3, 1.0),   # brilho alterado
    (0.0, 0.5),   # contraste alterado
]

# Nota: A FUNÇÃO DO OPENCV IMPLEMENTA A SEGUINTE FÓRMULA:
# img_cv2 = imagem * alpha + beta
# onde alpha é o fator de contraste e beta é o fator de brilho.

bins = 256
for beta, k in parametros_teste:
    img_func = alterar_brilho_contraste(imagem, beta, k)
    img_cv2 = cv2.convertScaleAbs(imagem, alpha=k*255, beta=beta*255)

    img_func_255 = np.clip(img_func * 255.0, 0, 255).astype(np.uint8)
    
    # Ravel transforma a matriz 2D (ou 3D) em um vetor 1D, o que é necessário para o histograma. 
    # Ex: M = [[1, 2], [3, 4]] -> M.ravel() = [1, 2, 3, 4],
    # assim, o histograma conta a frequência de cada intensidade de pixel em toda a imagem, 
    # independentemente da posição ou canal de cor.
    img_func_flat = img_func_255.ravel()
    img_cv2_flat = img_cv2.ravel()
    
    plt.figure(figsize=(18, 5))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Análise de Filtro - Beta (Brilho): {beta} | k (Contraste): {k}", fontsize=16, fontweight='bold')
    
    # --- SUBPLOT 1: Sua Função ---
    if img_func_255.ndim == 3:
        axes[0].imshow(img_func_255[:, :, ::-1])
    else:
        axes[0].imshow(img_func_255, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title("Função Personalizada")
    axes[0].axis('off') # Esconde os eixos
    
    # --- SUBPLOT 2: OpenCV ---
    if img_cv2.ndim == 3:
        axes[1].imshow(img_cv2[:, :, ::-1])
    else:
        axes[1].imshow(img_cv2, cmap='gray', vmin=0, vmax=255)
    axes[1].set_title("cv2.convertScaleAbs")
    axes[1].axis('off')
    
    # --- SUBPLOT 3: Histograma Comparativo ---
    axes[2].hist(img_func_flat, bins=bins, range=(0, 256), color='blue', alpha=0.5, label='Função Personalizada')
    axes[2].hist(img_cv2_flat, bins=bins, range=(0, 256), color='red', alpha=0.5, label='cv2.convertScaleAbs')
    axes[2].set_title("Comparação de Histogramas")
    axes[2].set_xlabel("Intensidade de Pixel (0 a 255)")
    axes[2].set_ylabel("Número de Pixels")
    axes[2].legend()
    
    # Ajusta o layout para nada ficar espremido
    plt.tight_layout()
    # Mostra a figura desta iteração do loop (as 3 imagens juntas)
    plt.show()

Nota: a função do openCV implementa a seguinte fórmula:
$
    \begin{equation}
    O(i,j) = \alpha \cdot I(i,j) + \beta
    \end{equation}
$
Em que:
- $O(i,j)$: É o pixel de saída;
- $\alpha$: É o fator de contraste;
- $I(i,j)$: pixel original da imagem;
- $\beta$: É o fator de brilho;

Fórmula implementada na função:
$
    \begin{equation}
        img\_contraste = k\cdot(imagem\_original - média) + média
    \end{equation}
$
Em que:
- $img\_contraste$: É a imagem de saída;
- $k$: É o fator de contraste;
- $media$: É um vetor com média relativa a cada uma das cores;

## Resposta para a questão 2:
Não. A função implementada segue uma lógica diferente daquela utilizada na biblioteca OpenCV. Enquanto a função de contraste proposta centraliza os dados em relação à média das intensidades, a função nativa do OpenCV os centraliza em relação a zero. A diferença prática consiste em:
- Ao centralizar em relação à média: Preservamos a luminosidade base da imagem. O contraste atua afastando os valores dessa média central, ou seja, os pixels escuros tornam-se mais escuros e os claros, mais claros, mantendo o equilíbrio.
- Ao centralizar em relação a zero: O "pivô" do alongamento passa a ser o preto (0). Ao aplicar o ganho de contraste, todos os valores da matriz são multiplicados e empurrados para a extremidade superior da escala. Isso gera um desvio na luminosidade média, culminando frequentemente em imagens esbranquiçadas e na saturação dos pixels mais claros.

Essa diferença é evidenciada principalmente no caso 3:
- Curva Azul (Função implementada): O histograma encolheu horizontalmente, mas a sua massa de dados continuou concentrada na mesma região central da distribuição original (em torno da intensidade 120-130).
- Curva Vermelha (OpenCV): O histograma também encolheu (refletindo a perda de contraste), mas foi inteiramente puxado para a esquerda, em direção ao zero. Isso comprova que o pivô matemático do OpenCV é o zero, o que resulta em uma imagem significativamente mais escura do que a original.

A diferença nos níveis de brilho se deve a fatores de arredondamento, que são tratados de forma diferente em cada método.

## Questão 3 - Cores
Dada uma imagem colorida, projete cada pixel (R,G,B) no plano de luminância constante $Y = \hat{Y}$ (média da imagem). Implemente e compare duas estratégias de projeção: 
- (a) via escalonamento linear do vetor original
- (b) via projeção ortogonal ao plano de luminância. Discuta qual técnica preserva melhor a identidade visual do cartoon e como lidar com o estouro de gamut (valores
fora de [0, 255]).


In [ ]:
imagem = cv2.imread("imagem/van_gogh.png", cv2.IMREAD_COLOR)

# Convertemos a imagem para float32 para evitar problemas de overflow durante as operações de soma e média
img = imagem.astype(np.float32)

# Y_sum = [
# [soma(r_11, g_11, b_11), soma(r_12, g_12, b_12), ..., soma(r_1n, g_1n, b_1n)],
# [soma(r_21, g_21, b_21), soma(r_22, g_22, b_22), ..., soma(r_2n, g_2n, b_2n)],
# ...
# [soma(r_m1, g_m1, b_m1), soma(r_m2, g_m2, b_m2), ..., soma(r_mn, g_mn, b_mn)]
# ]
Y_sum = np.sum(img, axis=2)


# Y_medio = soma_r + soma_g + soma_b / (m * n) = np.mean(Y_sum)

Y_medio = np.mean(Y_sum)

# Buscamos as novas soluções projetadas (r', g', b') para cada pixel original (i, j) 
# que satisfaçam a restrição do plano de luminância constante:
# r'_ij + g'_ij + b'_ij = Y_medio

# ==============================================================================
# (a) Escalonamento linear: 
# O plano R + G + B = Y_medio intercepta o eixo Z no ponto (0, 0, Y_medio), 
# o eixo X no ponto (Y_medio, 0, 0) e o eixo Y no ponto (0, Y_medio, 0). 
# Dessa forma, a porção visível desse plano forma um triângulo no primeiro 
# octante do espaço RGB (onde r, g, b >= 0). 
#
# Para cada pixel, encontramos a nova cor projetada "esticando" ou "encolhendo" 
# o vetor de cor original a partir da origem (0,0,0) até que ele toque esse plano.
# Fazemos isso escalonando os valores originais de r, g e b por um fator 'k', 
# proporcional à média alvo Y_medio em relação à soma original Y_sum[i, j]:
#
# Equação: k * (r_ij + g_ij + b_ij) = Y_medio 
# Substituindo: k * Y_sum[i, j] = Y_medio  =>  k = Y_medio / Y_sum[i, j]
#
# Novos valores: r' = k*r, g' = k*g, b' = k*b.
# ==============================================================================

k = np.zeros_like(Y_sum)

# A mascara serve para evitar divisão por zero nos pixels onde Y_sum é zero (preto)
mascara = Y_sum > 0

# Calculamos o fator de escalonamento k apenas para os pixels onde Y_sum é maior que zero
k[mascara] = Y_medio / Y_sum[mascara]

# k é uma matriz 2D com o mesmo tamanho da imagem, onde cada elemento representa o fator de escalonamento para aquele pixel específico.
# Para aplicar esse fator a cada canal de cor (r, g, b), precisamos expandir k para ter uma dimensão extra que corresponda aos canais de cor.
img_a_float = img * k[:, :, np.newaxis]
img_a_final = np.clip(img_a_float, 0, 255).astype(np.uint8)

# ==============================================================================
# (b) Projeção ortogonal:
# Para cada pixel, projetamos ortogonalmente o vetor de cor original (r, g, b) no plano definido por R + G + B = Y_medio. 
# Sabemos que o vetor normal a esse plano é n = (1, 1, 1). A projeção ortogonal de um ponto P = (r, g, b) no plano é dada por:
# P_proj = P + d * n = (r + d, g + d, b + d), onde d é a distância do ponto P ao plano, calculada por:
# d = [(R - r) + (G - g) + (B - b)] / ||n||^2 = (Y_medio - (r + g + b)) / 3 = (Y_medio - Y_sum[i, j]) / 3
# => P_proj = (r + d, g + d, b + d) = (r + (Y_medio - Y_sum[i, j]) / 3, g + (Y_medio - Y_sum[i, j]) / 3, b + (Y_medio - Y_sum[i, j]) / 3)
# ==============================================================================

deslocamento = (Y_medio - Y_sum) / 3
img_b_float = img + deslocamento[:, :, np.newaxis]
img_b_final = np.clip(img_b_float, 0, 255).astype(np.uint8)

img_original_rgb = imagem[:, :, ::-1] 
img_a_rgb = img_a_final[:, :, ::-1] 
img_b_rgb = img_b_final[:, :, ::-1] 

# Configurando o tamanho da figura para exibir lado a lado
plt.figure(figsize=(15, 5))

# Imagem 1: Original
plt.subplot(1, 3, 1) # (1 linha, 3 colunas, posição 1)
plt.imshow(img_original_rgb)
plt.title("1. Imagem Original")
plt.axis('off') # Esconde os eixos numéricos

# Imagem 2: Escalonamento Linear
plt.subplot(1, 3, 2)
plt.imshow(img_a_rgb)
plt.title("2. Escalonamento Linear (a)")
plt.axis('off')

# Imagem 3: Projeção Ortogonal
plt.subplot(1, 3, 3)
plt.imshow(img_b_rgb)
plt.title("3. Projeção Ortogonal (b)")
plt.axis('off')

# Calcula os espaços entre as imagens
plt.tight_layout()
plt.show()

## Resposta para a questão 3
1. Qual técnica preserva melhor a identidade visual?
- Matematicamente, é o escalonamento linear. Isso se deve ao fato de que ele preserva a proporção de distribuição dos pixels, mantendo o matiz e a saturação da imagem original, alterando somente a luminosidade e/ou intensidade. No entanto, em algumas imagens (como a de van_gogh, presente na pasta de imagens no repositório) esse resultado pode se inverter, especialmente quando a diferença entre a cor original e a média alvo causa um grande estouro de gamut (valores acima de 255). Como o escalonamento linear é multiplicativo, pixels muito escuros são forçados a atingir uma média de luminância alta e recebem um multiplicador maior. Ao aplicar o clipping para forçar esses valores de volta ao limite de 255, a proporção original entre os canais R, G e B deixa se perde, gerando manchas distorcidas. Nesses casos extremos, a projeção ortogonal, por realizar um deslocamento aditivo constante, pode evitar saturações tão agressivas e gerar um resultado visualmente mais suave, mesmo sacrificando parte da saturação original. Na imagem em questão (de van_gogh), vemos que os traços são mais preservados com a projeção ortogonal.

2. Como lidar com o estouro de gamut (valores fora de [0, 255])?
- A abordagem mais direta e a utilizada na implementação foi o "clipping", fixando os valores menores que 0 em 0 e os maiores que 255 em 255. Embora isso garanta a validade da imagem final para exibição, tem o custo de perder textura e distorcer cores nas áreas recortadas. Uma alternativa seria realizar uma normalização global de todos os pixels da imagem projetada para que caibam no intervalo $[0, 255]$, tornando o pixel de menor intensidade (mesmo que seja "negativa") como o novo 0 e o de maior intensidade (mesmo que passe de 255) como o novo 255. Porém, fazer esse remapeamento global alteraria a luminância média final da imagem, violando a restrição inicial do problema de manter os pixels no plano exato de $Y = \bar{Y}$. Portanto, o clipping é o método necessário para respeitar a restrição.


## Questão 4 - Filtros
Escreva uma função para efetuar a convolução de uma imagem com um filtro.
A função deve ter como entrada:
- Uma imagem em preto e branco.
- A matriz correspondente ao filtro

A saída deve ser a imagem filtrada. Teste com os seguintes filtros:
- Constante 3x3
- Derivada na horizontal 1x3 e derivada na vertical 3x1.
- Filtro de Sobel horizonta e vertical 3x3 (que tal olhar o módulo do gradiente também?)
- Filtro gaussiano com média zero e variância de 2 pixels, truncado numa matriz 5x5

As imagens para teste são à sua escolha, mas um dos testes deve ser com uma imagem de um tabuleiro de xadrez.

In [ ]:
# A aplicação de filtros consiste em processar a imagem original pixel a pixel, utilizando uma máscara 
# (ou kernel) para calcular o valor de cada pixel filtrado com base nos valores dos pixels vizinhos.
# Dessa forma, idealmente a matriz de filtragem (M:nxn) deve ser menor que a imagem e, normalmente, ter n impar. 
# o processo de filtragem envolve iterar sobre cada pixel da imagem. Dessa forma, para cada pixel (i, j), aplicamos a máscara M centrada nesse pixel, 
# multiplicando os valores da máscara pelos valores dos pixels vizinhos correspondentes e somando os resultados para obter o valor do pixel filtrado.
# Devemos tratar as bordas da imagem, onde a máscara pode ultrapassar os limites. Uma abordagem comum é usar preenchimento (padding) para lidar com essas situações,
# preenchendo a borda da imagem com zeros ou replicando os valores dos pixels mais próximos.

imagem = cv2.imread("imagem/PoucoContraste.png", cv2.IMREAD_GRAYSCALE)

def filtro(imagem, M):
    img = imagem.copy()
    img_alt, img_larg = img.shape
    m_alt, m_larg = M.shape
    
    # A máscara de convolução é rotacionada em 180 graus para garantir que a operação de filtragem seja realizada corretamente, 
    # seguindo a definição matemática de convolução. A rotação é necessária porque a operação de convolução envolve a aplicação 
    # da máscara de forma invertida em relação à imagem original.

    # np.flip(M, (0, 1)) inverte a ordem dos elementos da matriz M tanto na direção vertical (eixo 0) quanto na direção horizontal (eixo 1), isto é, 
    # Se M = [[a, b, c],
    #         [d, e, f],
    #         [g, h, i]]
    # Então np.flip(M, 0) resultará em:
    # M = [[g, h, i],
    #      [d, e, f],
    #      [a, b, c]]
    # E np.flip(M, 1) resultará em:
    # M = [[c, b, a],
    #      [f, e, d],
    #      [i, h, g]]
    # Assim, np.flip(M, (0, 1)) resultará em:
    # M = [[i, h, g],
    #      [f, e, d],
    #      [c, b, a]]

    M = np.flip(M, (0, 1))

    # // retorna a parte inteira da divisão, ou seja, o número de vezes que m_alt cabe em img_alt
    # calculamos as margens necessárias, em outras palavras, para garantir que a máscara possa ser aplicada a todos os pixels, incluindo as bordas.
    pad_alt = m_alt // 2
    pad_larg = m_larg // 2

    # img_padded = np.pad(img, ((cima, baixo), (esquerda, direita)), mode='constant', constant_values=0)
    img_padded = np.pad(img, ((pad_alt, pad_alt), (pad_larg, pad_larg)), mode='constant', constant_values=0)

    # Matriz inicial como float para suportar valores negativos
    imagem_filtrada = np.zeros(img.shape, dtype=float)

    for i in range(img_alt):
        for j in range(img_larg):
            # Extrai a região da imagem correspondente à máscara
            regiao = img_padded[i:i+m_alt, j:j+m_larg]
            # Realizando a convolução: multiplicando elemento a elemento a região da imagem pela máscara rotacionada e 
            # somando os resultados para obter o valor do pixel filtrado.
            valor_filtrado = np.sum(regiao * M)
            imagem_filtrada[i, j] = valor_filtrado
    
    # Após o processo de filtragem, os valores resultantes podem estar fora do intervalo [0, 255] e podem ser negativos
    # np.clip limita os valores entre 0 e 255, e .astype(np.uint8) converte para o tipo de dado apropriado

    imagem_filtrada = np.clip(imagem_filtrada, 0, 255).astype(np.uint8)
    return imagem_filtrada

# filtro constante 3x3
filtro_1 = (1/9)*np.array([
    [ 1, 1, 1],
    [ 1, 1, 1],
    [ 1, 1, 1]
])
# filtro_1 = np.array([
#     [ 0, -1, 0],
#     [-1, 4, -1],
#     [ 0, -1, 0]
# ])
# Derivada na Horizontal
# Funciona da seguinte forma: para cada pixel, subtraímos o valor do pixel à esquerda do valor do pixel à direita. Como o nome diz,
# se enxergarmos cada pixel como um ponto f(x,y), a derivada na horizontal representa a taxa de variação da intensidade de pixel ao longo do eixo x, 
# ou seja, como a intensidade muda quando nos movemos horizontalmente pela imagem.
filtro_2 = np.array([
    [-1, 0, 1]
])

# Derivada na Vertical
# Funciona de forma similar à derivada horizontal, mas agora subtraímos o valor do pixel acima do valor do pixel abaixo.
# Assim, a derivada na vertical representa a taxa de variação da intensidade de pixel ao longo do eixo y, ou seja, 
# como a intensidade muda quando nos movemos verticalmente pela imagem.
filtro_3 = np.array([
    [-1],
    [ 0],
    [ 1]
])

# Filtro de Sobel na Horizontal
filtro_4 = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

# Filtro de Sobel na Vertical
filtro_5 = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
])

# A fórmula da Gaussiana com média 0 é dada por:
# G(x, y) = (1 / (2 * pi * delta^2)) * exp(-(x^2 + y^2) / (2 * delta^2))
# Como a variância é de 2px, temos delta^2 = 2.
# Como devemos trucar para uma máscara de 5x5, os valores de x e y variam de -2 a 2:
# x,y pertence a {-2, -1, 0, 1, 2}
# Se K(x, y) = exp(-(x^2 + y^2) / (2 * delta^2)), então a máscara de 5x5 é dada por:

# Filtro Gaussiano com média 0 e variância de 2px, truncado para 5x5
kernel = cv2.getGaussianKernel(ksize=5, sigma=np.sqrt(2))
# kernel é um vetor coluna de 5x1, e para obter a máscara 5x5, fazemos o produto externo do kernel consigo mesmo, ou seja, kernel @ kernel.T
filtro_6 = kernel @ kernel.T

filtros = [filtro_1, filtro_2, filtro_3, filtro_4, filtro_5, filtro_6]
nomes_filtros = [
    "Constante 3x3", 
    "Derivada Horizontal", 
    "Derivada Vertical", 
    "Sobel Horizontal", 
    "Sobel Vertical",
    "Gaussiano 5x5"
]

lista_de_imagens = [
    "PoucoContraste.png",
    "xadrez.png"
]

for figura in lista_de_imagens:
    imagem = cv2.imread(f"imagem/{figura}", cv2.IMREAD_GRAYSCALE)

    plt.figure(figsize=(15, 10))
    plt.suptitle(f"Análise de Filtros: {figura}", fontsize=16)

    # 1. Plotando a imagem original
    plt.subplot(3, 3, 1)
    plt.imshow(imagem, cmap='gray')
    plt.title("Imagem Original")
    plt.axis('off')

    resultado_sobel_horizontal = None
    resultado_sobel_vertical = None

    for i, (filtro_i, nome) in enumerate(zip(filtros, nomes_filtros)):
        img_resultado = filtro(imagem, filtro_i)

        if nome == "Sobel Horizontal":
            resultado_sobel_horizontal = img_resultado
        elif nome == "Sobel Vertical":
            resultado_sobel_vertical = img_resultado
        
        plt.subplot(3, 3, i + 2)
        plt.imshow(img_resultado, cmap='gray')
        plt.title(nome)
        plt.axis('off')
        
    if resultado_sobel_horizontal is not None and resultado_sobel_vertical is not None:
        # O módulo do gradiente é calculado usando a função np.hypot, que computa a raiz quadrada 
        # da soma dos quadrados dos dois resultados de Sobel, ou seja, sqrt(sobel_horizontal^2 + sobel_vertical^2),
        modulo = np.hypot(resultado_sobel_horizontal, resultado_sobel_vertical)
        modulo = np.clip(modulo, 0, 255).astype(np.uint8)
        
        # Plota o módulo no último espaço do grid (posição 8)
        plt.subplot(3, 3, 8) 
        plt.imshow(modulo, cmap='gray')
        plt.title("Módulo do Gradiente (Sobel)")
        plt.axis('off')

    plt.tight_layout()
    # O plt.show() pausa a execução do loop. Quando você fechar a janela da figura, 
    # o loop continua e a próxima imagem será processada e exibida.
    plt.show()

## Questão 5 - Redução de dimensão

Faça a redução de uma imagem pela metade do número de linhas e metade do número de colunas de duas formas:
- Sem fazer suavização, apenas cortando linhas e colunas
- Com a suavização antes de eliminar linhas e colunas

In [ ]:
def reducao_imagem(imagem, s, fator_reducao):
    """
    se s = 0, imagem sem suavização
    se s = 1, imagem suavizada
    """
    f = fator_reducao
    img = imagem.copy()
    alt, larg = img.shape[:2]
    nova_alt = alt // f
    nova_larg = larg // f
    
    imagem_reduzida = np.zeros((nova_alt, nova_larg, img.shape[2]), dtype=img.dtype)
    if s == 1:
        # Usando o desfoque Gaussiano com uma janela 3x3
        img = cv2.GaussianBlur(img, (3, 3), 0)


    for c in range(img.shape[2]):
        for i in range(nova_alt):
            for j in range(nova_larg):
                imagem_reduzida[i, j, c] = img[f*i, f*j, c]
    
    # alternativamente, poderíamos usar a função de slicing do NumPy para reduzir a imagem sem precisar de loops explícitos:
    # imagem_reduzida = img[::f, ::f, :]
        
    return imagem_reduzida


lista_de_imagens = [
    "PoucoContraste.png",
    "xadrez.png",
    "snoopy.jpg",
    "van_gogh.png",
    "reliquias.png"
]

fator = 2
num_linhas = len(lista_de_imagens)

plt.figure(figsize=(12, 5 * num_linhas))
plt.suptitle("Comparação de Redução de Dimensão: Com vs Sem Suavização", fontsize=16)

for i, figura in enumerate(lista_de_imagens):
    imagem = cv2.imread(f"imagem/{figura}")
    
    # gerando as duas versões da imagem
    img_sem_suavizacao = reducao_imagem(imagem, s=0, fator_reducao=fator)
    img_com_suavizacao = reducao_imagem(imagem, s=1, fator_reducao=fator)
    posicao_esquerda = (2 * i) + 1
    posicao_direita  = (2 * i) + 2
    if imagem.shape[2] == 3:
        img_sem_suavizacao = img_sem_suavizacao[:, :, ::-1]
        img_com_suavizacao = img_com_suavizacao[:, :, ::-1]

    # Plotando o resultado SEM suavização (Letra a)
    plt.subplot(num_linhas, 2, posicao_esquerda)
    plt.imshow(img_sem_suavizacao, cmap='gray')
    plt.title(f"{figura} - Sem Suavização", fontsize=12)
    plt.axis('off')

    # Plotando o resultado COM suavização (Letra b)
    plt.subplot(num_linhas, 2, posicao_direita)
    plt.imshow(img_com_suavizacao, cmap='gray')
    plt.title(f"{figura} - Com Suavização", fontsize=12)
    plt.axis('off')

# ajustando o espaçamento para que os títulos fiquem organizados
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()